In [5]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

# Helper function to simulate the system
def simulate_system(
    c=5,
    battery_capacity_Wh=30,
    num_batteries=1,
    sleep_power_W=0.1,
    measuring_power_W=1,
    measuring_time_min=1,
    sending_power_W=10,
    sending_time_min=1,
    num_solar_cells=1,
    solar_spring_Wh=1,
    solar_summer_Wh=1,
    solar_autumn_Wh=1,
    solar_winter_Wh=1,
    simulation_days=365
):
    # Total battery capacity
    total_battery_capacity_Wh = battery_capacity_Wh * num_batteries
    # Time resolution: 1 min
    dt = 1/60  # hours
    total_minutes = simulation_days * 24 * 60
    time_hours = np.arange(total_minutes) / 60
    
    # Determine solar power per hour based on season
    # Seasons: Spring (Mar-May), Summer (Jun-Aug), Autumn (Sep-Nov), Winter (Dec-Feb)
    # Assume day 1 is Jan 1
    solar_power_W = np.zeros(total_minutes)
    for day in range(simulation_days):
        month = (day // 30) % 12 + 1  # crude month approximation
        if month in [3,4,5]:
            daily_solar_Wh = solar_spring_Wh * num_solar_cells
        elif month in [6,7,8]:
            daily_solar_Wh = solar_summer_Wh * num_solar_cells
        elif month in [9,10,11]:
            daily_solar_Wh = solar_autumn_Wh * num_solar_cells
        else:
            daily_solar_Wh = solar_winter_Wh * num_solar_cells

        # Solar hours 7-21 (14 hours)
        solar_hours = np.arange(5*60, 23*60)  # minutes in day
        # Normal distribution parameters
        mu = 14 * 60  # peak at 14:00 (in minutes)
        sigma = 3 * 60  # spread (3 hours stddev)
        gauss = np.exp(-0.5 * ((solar_hours - mu) / sigma) ** 2)
        gauss /= gauss.sum()  # normalize so sum = 1
        solar_profile_Wh = daily_solar_Wh * gauss  # scale to total daily energy
        solar_power_W[day*24*60 + solar_hours] = solar_profile_Wh
    
    # Convert Wh to W per minute
    solar_power_W = solar_power_W / 1  # already per hour; dt=1/60h considered in update

    # Initialize energy storage and consumption
    battery_energy_Wh = np.zeros(total_minutes)
    battery = total_battery_capacity_Wh
    measuring_energy = np.zeros(total_minutes)
    sending_energy = np.zeros(total_minutes)
    sleep_energy = np.zeros(total_minutes)
    solar_generated = np.zeros(total_minutes)
    
    # Cycle scheduling per day
    cycle_schedule = np.zeros(24*60)  # minutes
    minutes_per_cycle = int(measuring_time_min + sending_time_min)
    if c > 0:
        interval = 24*60 // c
        for i in range(c):
            start_minute = i*interval
            cycle_schedule[start_minute:start_minute+int(measuring_time_min)] = 1  # measuring
            cycle_schedule[start_minute+int(measuring_time_min):start_minute+minutes_per_cycle] = 2  # sending
    
    for t in range(total_minutes):
        # Determine state
        state = cycle_schedule[t % (24*60)]
        # Power draw in W
        if state == 1:
            draw = measuring_power_W
            measuring_energy[t] = draw * dt
        elif state == 2:
            draw = sending_power_W
            sending_energy[t] = draw * dt
        else:
            draw = sleep_power_W
            sleep_energy[t] = draw * dt
        
        # Solar generation
        gen = solar_power_W[t]
        solar_generated[t] = gen * dt
        
        # Battery update
        battery += (gen - draw) * dt
        # Cap battery
        if battery > total_battery_capacity_Wh:
            battery = total_battery_capacity_Wh
        if battery < 0:
            battery = 0
        battery_energy_Wh[t] = battery

    df = pd.DataFrame({
        'Time (h)': time_hours,
        'Battery Energy (Wh)': battery_energy_Wh,
        'Solar Generated (Wh)': solar_generated,
        'Measuring Energy (Wh)': measuring_energy,
        'Sending Energy (Wh)': sending_energy,
        'Sleep Energy (Wh)': sleep_energy
    })
    return df

# Function to update the plot
def update_plot(**kwargs):
    df = simulate_system(**kwargs)
    
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        subplot_titles=("Battery Energy", "Power Flow per Minute"))
    
    # Battery subplot
    fig.add_trace(go.Scatter(x=df['Time (h)'], y=df['Battery Energy (Wh)'], name='Battery Energy', line=dict(color='blue')), row=1, col=1)
    
    # Power flows subplot
    fig.add_trace(go.Scatter(x=df['Time (h)'], y=df['Solar Generated (Wh)'], name='Solar Generated', line=dict(color='orange')), row=2, col=1)
    fig.add_trace(go.Scatter(x=df['Time (h)'], y=df['Measuring Energy (Wh)'], name='Measuring Draw', line=dict(color='green')), row=2, col=1)
    fig.add_trace(go.Scatter(x=df['Time (h)'], y=df['Sending Energy (Wh)'], name='Sending Draw', line=dict(color='red')), row=2, col=1)
    fig.add_trace(go.Scatter(x=df['Time (h)'], y=df['Sleep Energy (Wh)'], name='Sleep Draw', line=dict(color='purple')), row=2, col=1)
    
    fig.update_layout(height=700, width=1000, title_text="Solar-Powered System Simulation", legend=dict(orientation="h"))
    fig.update_xaxes(title_text="Time [hours]", row=2, col=1)
    fig.update_yaxes(title_text="Energy [Wh]", row=1, col=1)
    fig.update_yaxes(title_text="Energy Flow [Wh/min]", row=2, col=1)
    
    fig.show()

# Sliders
interactive_plot = widgets.interactive(update_plot,
    c=widgets.IntSlider(min=0,max=48,step=1,value=5, description='Cycles/day'),
    battery_capacity_Wh=widgets.FloatSlider(min=1,max=100,step=1,value=30, description='Battery Wh'),
    num_batteries=widgets.IntSlider(min=1,max=10,step=1,value=1, description='Num Batteries'),
    sleep_power_W=widgets.FloatSlider(min=0,max=1,step=0.01,value=0.01, description='Sleep W'),
    measuring_power_W=widgets.FloatSlider(min=0,max=10,step=0.1,value=1, description='Measuring W'),
    measuring_time_min=widgets.IntSlider(min=1,max=60,step=1,value=1, description='Measuring min'),
    sending_power_W=widgets.FloatSlider(min=0,max=20,step=0.1,value=10, description='Sending W'),
    sending_time_min=widgets.IntSlider(min=1,max=60,step=1,value=1, description='Sending min'),
    num_solar_cells=widgets.IntSlider(min=1,max=10,step=1,value=8, description='Solar Cells'),
    solar_spring_Wh=widgets.FloatSlider(min=0,max=10,step=0.1,value=4, description='Solar Spring Wh'),
    solar_summer_Wh=widgets.FloatSlider(min=0,max=10,step=0.1,value=0.5, description='Solar Summer Wh'),
    solar_autumn_Wh=widgets.FloatSlider(min=0,max=10,step=0.1,value=1, description='Solar Autumn Wh'),
    solar_winter_Wh=widgets.FloatSlider(min=0,max=10,step=0.1,value=0.5, description='Solar Winter Wh'),
    simulation_days=widgets.IntSlider(min=1,max=365,step=1,value=4, description='Days')
)

display(interactive_plot)

interactive(children=(IntSlider(value=5, description='Cycles/day', max=48), FloatSlider(value=30.0, descriptio…